# 01 — Data Cleaning
Load the raw Kaggle CSV (`Hospital_Wait_Time_Data.csv`), select the relevant columns, and export a clean processed version for SQL/dashboard/ML use.

In [ ]:
import pandas as pd

# Load raw data (place the Kaggle CSV in data/raw/ first, keep the exact filename)
df = pd.read_csv('../data/raw/Hospital_Wait_Time_Data.csv')
print(df.shape)
df.head()

In [ ]:
# Basic inspection
print(df.dtypes)
print()
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
# Keep only the columns we actually need for this project.
# TriageToProviderStartTime = time (minutes) between triage completion and the
# doctor starting the consultation — this is our core "OPD wait time" metric.
cols = [
    'PatientID', 'AgeGroup', 'Department', 'AppointmentType', 'InsuranceType',
    'ArrivalMethod', 'VisitDate', 'ArrivalHour', 'DayOfWeek', 'IsWeekend',
    'Month', 'Quarter', 'TriageCategory', 'FacilityOccupancyRate',
    'ProvidersOnShift', 'NursesOnShift', 'StaffToPatientRatio',
    'TriageToProviderStartTime', 'TotalTimeInHospital', 'TotalDelayTime'
]
df = df[cols].copy()

# Rename our target column to something clearer for SQL/dashboard/ML
df = df.rename(columns={'TriageToProviderStartTime': 'WaitTimeMinutes'})

df.head()

In [ ]:
# Clean: drop duplicates, drop any rows with a missing/negative wait time
df = df.drop_duplicates()
df = df[df['WaitTimeMinutes'] > 0]

# Parse VisitDate (format: DD-MM-YYYY)
df['VisitDate'] = pd.to_datetime(df['VisitDate'], format='%d-%m-%Y')

print(df.shape)
df.describe(include='all').T.head(20)

In [ ]:
# Quick sanity-check visualization before exporting
import matplotlib.pyplot as plt

df.groupby('Department')['WaitTimeMinutes'].mean().sort_values(ascending=False).plot(kind='bar', figsize=(8,4), title='Avg Wait Time by Department (minutes)')
plt.ylabel('Minutes')
plt.tight_layout()
plt.show()

In [ ]:
# Export cleaned data for SQL import and dashboard use
df.to_csv('../data/processed/opd_visits_clean.csv', index=False)
print('Saved cleaned dataset:', df.shape)